# Wan 2.1 Text-to-Video Production
Setup ComfyUI no Google Colab com Wan 2.1 para geração de vídeos

## 1. Instalação do ComfyUI e dependências

In [1]:
%cd /content
print("-= Initial setup ComfyUI =-")
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git pull
print("-= Install dependencies =-")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q av

/content
-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 38016, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 38016 (delta 65), reused 38 (delta 33), pack-reused 37922 (from 2)
Receiving objects: 100% (38016/38016), 80.31 MiB | 26.48 MiB/s, done.
Resolving deltas: 100% (25531/25531), done.
/content/ComfyUI
Already up to date.
-= Install dependencies =-
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.9/262.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 11.8 MB/s 

## 2. Instalar custom nodes para Wan 2.1

In [2]:
print("-= Installing Wan 2.1 custom nodes =-")
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-Wan.git
print("\n✓ Custom nodes instalados")

-= Installing Wan 2.1 custom nodes =-
/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI-Wan'...
fatal: could not read Username for 'https://github.com': No such device or address

✓ Custom nodes instalados


## 3. Baixar modelos Wan 2.1 (1.3B)Model sizes: 1.3B (~2.5GB) funciona bem na T4/L4

In [3]:
%cd /content/ComfyUI
import os
os.makedirs('models/unet', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)
os.makedirs('models/clip', exist_ok=True)
print("-= Downloading Wan 2.1 T2V 1.3B model =-")
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors -O models/unet/wan2.1_t2v_1.3B_bf16.safetensors
print("-= Downloading VAE =-")
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/vae/wan_2.1_vae.safetensors -O models/vae/wan_2.1_vae.safetensors
print("-= Downloading CLIP =-")
!wget -q https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -O models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors
print("✓ Modelos baixados!")

/content/ComfyUI
-= Downloading Wan 2.1 T2V 1.3B model =-
-= Downloading VAE =-
-= Downloading CLIP =-
✓ Modelos baixados!


## 4. Iniciar ComfyUI

In [4]:
%cd /content/ComfyUI
import threading, time
from IPython.display import display, HTML

def run():
    import subprocess
    subprocess.run(['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'])

threading.Thread(target=run, daemon=True).start()
time.sleep(10)
print("ComfyUI starting...")
display(HTML('<a href="/proxy/8188" target="_blank">Abrir ComfyUI</a>'))

/content/ComfyUI
ComfyUI starting...


## 5. Workflow JSON - copie e cole no ComfyUI

In [5]:
import json
workflow = {
  "last_node_id": 47,
  "last_link_id": 93,
  "nodes": [
    {"id": 38, "type": "CLIPLoader", "pos": [12, 184], "size": [390, 82], "widgets_values": ["umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default"]},
    {"id": 37, "type": "UNETLoader", "pos": [485, 57], "size": [346, 82], "widgets_values": ["wan2.1_t2v_1.3B_bf16.safetensors", "default"]},
    {"id": 39, "type": "VAELoader", "pos": [866, 499], "size": [306, 58], "widgets_values": ["wan_2.1_vae.safetensors"]},
    {"id": 40, "type": "EmptyHunyuanLatentVideo", "pos": [520, 620], "size": [315, 130], "widgets_values": [832, 480, 33, 1]},
    {"id": 6, "type": "CLIPTextEncode", "pos": [415, 186], "size": [422, 164], "widgets_values": ["a fox moving quickly in a beautiful winter scenery nature trees sunset tracking camera"], "color": "#232", "bgcolor": "#353"},
    {"id": 7, "type": "CLIPTextEncode", "pos": [413, 389], "size": [425, 180], "widgets_values": ["Overexposure, static, blurred details, subtitles, paintings, pictures, still, overall gray, worst quality, low quality, JPEG compression residue, ugly, mutilated, redundant fingers, poorly painted hands, poorly painted faces, deformed, disfigured, deformed limbs, fused fingers, cluttered background, three legs, a lot of people in the background, upside down"], "color": "#322", "bgcolor": "#533"},
    {"id": 3, "type": "KSampler", "pos": [863, 187], "size": [315, 262], "widgets_values": [878361741413693, "randomize", 30, 6, "uni_pc", "simple", 1]},
    {"id": 8, "type": "VAEDecode", "pos": [1210, 190], "size": [210, 46]},
    {"id": 47, "type": "SaveWEBM", "pos": [2367, 193], "size": [315, 130], "widgets_values": ["ComfyUI", "vp9", 24, 32]},
    {"id": 28, "type": "SaveAnimatedWEBP", "pos": [1460, 190], "size": [870, 643], "widgets_values": ["ComfyUI", 16, False, 90, "default"]}
  ],
  "links": [
    [35, 3, 0, 8, 0, "LATENT"],
    [46, 6, 0, 3, 1, "CONDITIONING"],
    [52, 7, 0, 3, 2, "CONDITIONING"],
    [56, 8, 0, 28, 0, "IMAGE"],
    [74, 38, 0, 6, 0, "CLIP"],
    [75, 38, 0, 7, 0, "CLIP"],
    [76, 39, 0, 8, 1, "VAE"],
    [91, 40, 0, 3, 3, "LATENT"],
    [92, 37, 0, 3, 0, "MODEL"],
    [93, 8, 0, 47, 0, "IMAGE"]
  ]
}
with open('/content/ComfyUI/wan21_workflow.json', 'w') as f:
    json.dump(workflow, f, indent=2)
print("Workflow salvo em: /content/ComfyUI/wan21_workflow.json")

Workflow salvo em: /content/ComfyUI/wan21_workflow.json
